# 🧠 NB-03: LoRA Fine-tuning – Chain-of-Thought Distillation
**Goal:** Use PEFT/LoRA to fine-tune a small LLM on Claude's `<think>` reasoning chains.

**Modality:** Text (CoT) | **Model:** TinyLlama via PEFT LoRA

In [ ]:
!pip install peft transformers datasets bitsandbytes accelerate -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
import torch

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","v_proj"]
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

# Train on user → <think>reasoning</think> + response
cot_texts = []
for d in data:
    if d["thinking"]:
        txt = f"### Question:\n{d['user']}\n\n### Thinking:\n{d['thinking'][:400]}\n\n### Answer:\n{d['response'][:400]}{tokenizer.eos_token}"
        cot_texts.append(txt)

def tok(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=512, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

ds = Dataset.from_dict({"text": cot_texts}).map(tok, batched=True, remove_columns=["text"]).train_test_split(0.1)

args = TrainingArguments(output_dir="./lora-cot", num_train_epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    fp16=torch.cuda.is_available(), logging_steps=10, report_to="none")
Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"]).train()
model.save_pretrained("./lora-cot-adapter")
print("✅ LoRA CoT adapter saved!")
